# 3 - Fine-tune DroneMegaDetector for birds

Closing DroneMegaDetector's one documented gap.

## Why this one is harder

DroneMegaDetector's `animal` class is effectively `mammal` - the repo says
behaviour on non-mammals is undefined, and the preprint reports **weaker
transfer to unseen avian imagery**. This is a known, stated limitation, not
something we are guessing at.

So unlike the seal notebook, expect to be starting from a low baseline.
Everything in the model's training data is a large body on open ground; a
wing silhouette from above is a different shape, often smaller, and often
over water.

Two consequences:

- **You need more data here than for seals.** A few hundred frames may not be
  enough to teach a genuinely new appearance class.
- **Watch the regression check especially closely.** Pushing the `animal`
  class to cover birds as well as mammals is asking it to widen, and widening
  usually costs precision somewhere.

The project's own taxonomy puts `SEABIRD` in a separate response group from
`PINNIPED` precisely because they are different rescues - so a dedicated bird
model fits how the dispatch side already thinks.

## The one design decision, stated up front

**This fine-tune does not add a class.** The output stays
`person, animal, vehicle` in DroneMegaDetector's order. A bird model
is not a model with a `bird` class; it is a model whose `animal` class
finds bird reliably.

Three reasons:

1. **Every checkpoint stays interchangeable.** One decode path, one
   `CLASS_NAMES` in `backend/lifejacket/drones/detector.py`, and switching
   models in production is three environment variables.
2. **Species naming is already somebody else's job.** The identification
   agent does it on a cropped frame, through `taxonomy.py`, which pools
   probabilities up the taxonomic tree. The detector does not need to compete
   with that.
3. **It avoids the `num_classes` trap.** Changing the head size on a
   fine-tune is a known source of silent misconfiguration in RF-DETR.

There is a cell near the end that shows what changing this would involve, if
you decide you want it.

In [ ]:
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dmd
import matplotlib.pyplot as plt
from dmd import load_coco, load_model, predict_images

HERE = Path.cwd().parent
DATA = Path.home() / "aerial-data"

def run(*args):
    """Call one of the sibling scripts and stream its output."""
    command = [sys.executable, *[str(a) for a in args]]
    print("$", " ".join(command[1:]), "\n")
    process = subprocess.run(command, cwd=HERE, text=True)
    if process.returncode != 0:
        raise SystemExit(f"failed: {' '.join(command[1:])}")

## Step 1 - label, then split by capture session

Export COCO from Label Studio or CVAT with the category names exactly
`person`, `animal`, `vehicle`. `dataset.py` matches on name and will stop if
anything else turns up.

**Splits are by capture session, never random.** Consecutive frames of one
animal are near-identical; split them randomly and the same animal lands in
train and test, so the model has effectively seen the answers and the score
means nothing.

Run `--inspect` first and read the session list.

In [ ]:
RAW = DATA / "birds_labelled"
SPLIT = DATA / "birds"

run(HERE / "dataset.py", "--coco", RAW / "_annotations.coco.json",
    "--images", RAW, "--out", SPLIT, "--inspect")

If the session keys above look like individual frames rather than capture
sessions, stop: your test split will leak. Fix it with `--group-regex`
before writing anything.

If a split came out empty, you do not have enough sessions. The script will
refuse to write rather than hand you a dataset that cannot train.

In [ ]:
run(HERE / "dataset.py", "--coco", RAW / "_annotations.coco.json",
    "--images", RAW, "--out", SPLIT, "--overwrite")

## Step 2 - look at the data before training on it

Cheap, and it catches the mistakes that are expensive later: boxes on the
wrong objects, a whole session mislabelled, targets far too small.

In [ ]:
train_set = load_coco(SPLIT / "train" / "_annotations.coco.json")
print(f"train: {len(train_set)} images, {train_set.box_count} boxes")
print("per class:", train_set.per_class_counts())

sizes = dmd.pixels_on_target([b for boxes in train_set.boxes.values() for b in boxes])
figure, axis = plt.subplots(figsize=(7, 3.5))
axis.hist(sizes, bins=30, color="#81C784", edgecolor="#555")
axis.axvline(32, color="#C62828", linestyle="--", label="~32 px detector floor")
axis.set_xlabel("longest side of a box (px)")
axis.legend()
plt.show()

sample = sorted(train_set.images)[:6]
dmd.show_grid(
    [dmd.draw_boxes(train_set.image_path(i), None, train_set.boxes.get(i, []))
     for i in sample],
    columns=3,
    titles=[train_set.image_path(i).name for i in sample],
)
plt.show()

## Step 3 - where is the model now, before training?

You cannot claim an improvement without this number.

In [ ]:
TEST = SPLIT / "test" / "_annotations.coco.json"

run(HERE / "evaluate.py", "--weights", "dmd", "--data", TEST, "--tag", "birds-v1-before")

## Step 4 - fine-tune, with replay

**Replay matters.** Fine-tuning only on bird makes the model better at
bird and measurably worse at everything else, including `person` and
`vehicle`, which every site needs. That is catastrophic forgetting.

`--replay DIR` mixes a slice of general data into the training set to hold it
off. Point it at any dataset in the same layout - your NOAA set, or a general
drone set. The validation split stays purely task-specific, because what you
want to know is whether the model got better at *this* job.

If you have no general set to hand, lower the learning rate to `5e-5` and
shorten training instead. Less adaptation, but less forgetting too.

In [ ]:
REPLAY = DATA / "general"     # set to None if you have nothing to mix in
EPOCHS = 60
LR = 5e-05

arguments = [HERE / "finetune.py", "--data", SPLIT, "--name", "birds-v1",
             "--epochs", EPOCHS, "--lr", LR]
if REPLAY and REPLAY.exists():
    arguments += ["--replay", REPLAY, "--replay-fraction", 0.25]
else:
    print("no replay set found - training on the specific data alone.")
    print("Watch the general-regression check in step 6 closely.\n")

run(*arguments, "--dry-run")      # check the layout first; drop --dry-run to train

The dry run checks the dataset layout, the class order and the base weights
without starting training. If it passed, train for real.

**On a GPU VM, run the real training in `tmux` from a terminal, not here.**
A notebook cell holding a multi-hour run will die when the browser proxy
drops, and Workbench's idle shutdown counts kernel activity, not shell
activity. The command is printed below - copy it.

In [ ]:
print("tmux new -s birds-v1")
print(f"cd {HERE}")
print("python finetune.py \\")
print(f"    --data {SPLIT} \\")
print(f"    --name birds-v1 --epochs {EPOCHS} --lr {LR} \\")
print("    --replay ~/aerial-data/general --replay-fraction 0.25")
print("\n# Ctrl-b d to detach, tmux attach -t birds-v1 to come back")

In [ ]:
# Or run it here anyway, for a short job. Uncomment to use.
# run(*arguments)

## Step 5 - did it get better at the job?

In [ ]:
RUN = HERE / "runs" / "birds-v1"
candidates = sorted(RUN.glob("checkpoint_best*.pth")) or sorted(RUN.glob("*.pth"))
BEST = candidates[0] if candidates else None
print("best weights:", BEST)

if BEST:
    run(HERE / "evaluate.py", "--weights", BEST, "--data", TEST,
        "--tag", "birds-v1-after",
        "--compare", HERE / "results" / "eval_birds-v1-before.json")

## Step 6 - did it get worse at everything else?

**Do not skip this.** A bird model that stopped detecting people is a
worse product in the field than stock DroneMegaDetector, and without this
check you would ship it not knowing.

`evaluate.py --compare` flags any metric that fell by 0.05 or more.

In [ ]:
GENERAL = DATA / "noaa_eval" / "_annotations.coco.json"

if BEST and GENERAL.exists():
    run(HERE / "evaluate.py", "--weights", BEST, "--data", GENERAL,
        "--tag", "birds-v1-general",
        "--compare", HERE / "results" / "eval_noaa-dmd.json")
else:
    print("No general evaluation set. Build one with notebook 1 before publishing.")

If that reported regressions: raise `--replay-fraction` to 0.4, or drop the
learning rate to `5e-5`, and train again. Both trade a little task accuracy
for retained general ability, which is almost always the right trade for a
dispatch tool.

## Step 7 - look at before and after, side by side

Numbers tell you whether; pictures tell you why.

In [ ]:
if BEST:
    test_set = load_coco(TEST)
    ids = sorted(test_set.images)[:4]
    paths = [test_set.image_path(i) for i in ids]

    before = predict_images(load_model(), paths, threshold=0.05)
    after = predict_images(load_model(BEST), paths, threshold=0.05)

    THRESHOLD = 0.3
    panels, titles = [], []
    for i, path in zip(ids, paths, strict=True):
        gt = test_set.boxes.get(i, [])
        panels.append(dmd.draw_boxes(path, before[path], gt, threshold=THRESHOLD))
        titles.append(f"{path.name} - stock DMD")
        panels.append(dmd.draw_boxes(path, after[path], gt, threshold=THRESHOLD))
        titles.append(f"{path.name} - birds-v1")
    dmd.show_grid(panels, columns=2, titles=titles)
    plt.show()

## Flocks are a different problem, not a harder version of this one

Worth knowing before you spend a week on it. Which case are you in?

**A few large birds on a beach** - an oiled pelican has a 2 m wingspan, which
at 50 m altitude is comparable to a sea lion. This notebook handles it, and
the fine-tune should work much like the seal one.

**A colony** - hundreds of terns per frame, each 10-20 px. That case needs:

- **Tiled inference.** DroneMegaDetector's own batch script slices into
  1280x1280 tiles with 100 px overlap and merges with NMS. `dmd.DMD_TILE_SIZE`
  and `DMD_TILE_OVERLAP` hold those values. Notebook 4 compares tiled against
  whole-frame.
- **A detection cap high enough.** Models commonly cap at 300 detections per
  image; `dmd.DMD_NUM_QUERIES` is 300 for this checkpoint, which is a hard
  ceiling on how many birds one frame can return no matter the threshold.
- **Targets above the floor.** 10-20 px is at or below what the model can
  resolve. Tiling helps precisely because each tile is upscaled.

Check which case you are in before deciding how much work this is.

In [ ]:
# How crowded are your frames, and are you anywhere near the query cap?
counts = [len(b) for b in train_set.boxes.values()]
if counts:
    figure, axis = plt.subplots(figsize=(7, 3.5))
    axis.hist(counts, bins=min(30, max(3, len(set(counts)))),
              color="#4FC3F7", edgecolor="#555")
    axis.axvline(dmd.DMD_NUM_QUERIES, color="#C62828", linestyle="--",
                 label=f"{dmd.DMD_NUM_QUERIES} query cap")
    axis.set_xlabel("boxes per frame")
    axis.legend()
    plt.show()
    print(f"max {max(counts)} boxes in one frame, median {sorted(counts)[len(counts) // 2]}")
    if max(counts) > dmd.DMD_NUM_QUERIES * 0.5:
        print("\nYou are in colony territory. Read the note above on tiling before training.")
    else:
        print("\nSparse frames - the ordinary fine-tune path is fine.")

## If birds should be their own class

Stronger argument here than for seals: "one large animal" and "a flock of
forty birds" are genuinely different dispatches, and the app's taxonomy
already treats `SEABIRD` as a separate response group with its own permits
and teams.

The change is four tuples and a test file:

1. `dmd.py` - `DMD_CLASSES = ("person", "animal", "vehicle", "bird")`
2. `backend/lifejacket/drones/detector.py` - the same tuple
3. `publish.py` - the same tuple
4. `class_map.yaml` - add `bird` to `targets`
5. `backend/tests/test_drone_detector.py` - decode tests assume three classes

Plus `num_classes=4` on the `RFDETRMedium` constructor in `finetune.py`
([issue #878](https://github.com/roboflow/rf-detr/issues/878)).

Still: measure first. If the three-class fine-tune already finds pelicans at
the recall you need, the fourth class buys information the identification
agent already provides.

## What good looks like here

Be realistic. Going from near-zero recall to 0.5 is a real success on a class
the base model was never trained for - it is not the 0.8 you might hope for.

- **Recall above zero at a usable precision** is the first milestone.
- **Check `person` did not regress.** Birds and distant people are both small
  dark blobs from above; teaching the model one can blur the other.
- **If recall stays near zero**, look at the pixels-on-target histogram again
  before adding epochs. Birds are small, and if they are under the floor this
  is a resolution problem.

An alternative worth benchmarking against: the
[DeepForest bird model](https://huggingface.co/weecology/deepforest-bird),
trained on 250,000+ annotations across 13 projects. If it beats your
fine-tune, use it - as a second detector for the bird case, routed by what
the coordinator is searching for.

## Step 8 - publish, if it earned it

Only publish a checkpoint that improved on its task **and** did not regress
on the general set.

```bash
python publish.py --weights runs/birds-v1/checkpoint_best_total.pth \
    --repo YOUR_USERNAME/lifejacket-bird --private
```

That prints a `DRONE_DETECTOR_*` block for `.env`. The app downloads by
pinned commit SHA and verifies the hash on every load.